In [1]:
import polars as pl
import os

In [2]:
if not os.path.exists("datasets"):
    os.mkdir("datasets")

In [6]:
# Buscamos el nombre del archivo parquet del libro

book_file = [f for f in os.listdir("datasets") if "df_book_10" in f and f.endswith(".parquet")][0]
path_book = os.path.join("datasets", book_file)

# Cargamos una vista previa
df_book_10 = pl.scan_parquet(path_book)

In [7]:
df_book_10.collect().head(5)

timestamp,local_timestamp,bids[0].price,bids[0].amount,asks[0].price,asks[0].amount,bids[1].price,bids[1].amount,asks[1].price,asks[1].amount,bids[2].price,bids[2].amount,asks[2].price,asks[2].amount,bids[3].price,bids[3].amount,asks[3].price,asks[3].amount,bids[4].price,bids[4].amount,asks[4].price,asks[4].amount,bids[5].price,bids[5].amount,asks[5].price,asks[5].amount,bids[6].price,bids[6].amount,asks[6].price,asks[6].amount,bids[7].price,bids[7].amount,asks[7].price,asks[7].amount,bids[8].price,bids[8].amount,asks[8].price,asks[8].amount,bids[9].price,bids[9].amount,asks[9].price,asks[9].amount
i64,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
1775001600571000,1775001601136017,68235.3,2.908,68235.4,6.377,68235.2,0.014,68235.5,0.02,68235.1,0.002,68235.6,0.015,68235.0,0.002,68235.7,0.067,68234.7,0.002,68235.8,0.152,68234.6,0.011,68235.9,0.187,68234.5,0.473,68236.4,0.001,68234.4,1.714,68236.5,0.002,68234.3,0.014,68236.6,0.002,68234.2,0.002,68236.8,0.003
1775001601141000,1775001601143545,68235.3,1.18,68235.4,6.377,68235.2,0.014,68235.5,0.02,68235.1,0.002,68235.6,0.015,68235.0,0.002,68235.7,0.067,68234.7,0.002,68235.8,0.152,68234.6,0.011,68235.9,0.187,68234.5,0.039,68236.4,0.001,68234.4,1.714,68236.5,0.002,68234.3,0.014,68236.6,0.002,68234.2,0.002,68236.8,0.003
1775001601193000,1775001601194988,68235.3,0.317,68235.4,7.1,68235.2,0.014,68235.5,0.02,68235.1,0.002,68235.6,0.015,68235.0,0.002,68235.7,0.067,68234.7,0.002,68235.8,0.152,68234.6,0.011,68235.9,0.187,68234.5,0.039,68236.4,0.001,68234.4,1.714,68236.5,0.002,68234.3,0.014,68236.6,0.002,68234.2,0.002,68236.8,0.003
1775001601245000,1775001601247772,68235.3,0.133,68235.4,7.106,68235.2,0.014,68235.5,0.02,68235.1,0.002,68235.6,0.015,68235.0,0.002,68235.7,0.163,68234.7,0.002,68235.8,0.152,68234.6,0.011,68235.9,0.187,68234.5,0.039,68236.4,0.001,68234.4,1.714,68236.5,0.002,68234.3,0.014,68236.6,0.002,68234.2,0.002,68236.8,0.003
1775001601297000,1775001601299697,68234.4,3.047,68234.5,13.275,68234.3,0.012,68234.6,0.004,68234.2,0.002,68235.3,1.237,68234.0,0.002,68235.4,1.645,68233.9,0.013,68235.5,0.018,68233.8,0.004,68235.6,0.015,68233.7,0.004,68235.7,0.015,68233.5,0.004,68235.8,0.152,68233.4,0.009,68235.9,0.045,68233.3,0.004,68236.4,0.001


In [8]:


def extract_fi2010_features_strict_polars(df_input: pl.DataFrame) -> pl.DataFrame:
    """Calcula las 144 características estrictas del paper clásico FI-2010.

    Elimina cualquier precio absoluto para garantizar la invarianza de escala.
    """
    # Aseguramos el orden temporal nativo
    df = df_input.sort("timestamp")

    exprs_b1 = []
    exprs_b2 = []
    exprs_b3 = []

    # -------------------------------------------------------------------------
    # BLOQUE 1: Datos Crudos del LOB (Variables 1 a 40)
    # -------------------------------------------------------------------------
    # Precios y volúmenes de los 10 niveles de Ask y Bid
    for n in range(10):
        exprs_b1.append(pl.col(f"asks[{n}].price").alias(f"v{n*4 + 1}"))
        exprs_b1.append(pl.col(f"asks[{n}].amount").alias(f"v{n*4 + 2}"))
        exprs_b1.append(pl.col(f"bids[{n}].price").alias(f"v{n*4 + 3}"))
        exprs_b1.append(pl.col(f"bids[{n}].amount").alias(f"v{n*4 + 4}"))

    # -------------------------------------------------------------------------
    # BLOQUE 2: Características Físicas / Geometría (Variables 41 a 90)
    # -------------------------------------------------------------------------

    # 1. Spreads por nivel (Variables 41 a 50) -> Ask_n - Bid_n
    for n in range(10):
        exprs_b2.append(
            (pl.col(f"asks[{n}].price") - pl.col(f"bids[{n}].price")).alias(
                f"v{41 + n}"
            )
        )

    # 2. Diferencias de precio respecto a la punta - ASK (Variables 51 a 60)
    # Mide la distancia desde el Ask del nivel 'n' hasta el Ask de la punta (Nivel 1)
    for n in range(10):
        exprs_b2.append(
            (pl.col(f"asks[{n}].price") - pl.col("asks[0].price")).alias(
                f"v{51 + n}"
            )
        )

    # 3. Diferencias de precio respecto a la punta - BID (Variables 61 a 70)
    # Mide la distancia desde el Bid de la punta (Nivel 1) hasta el Bid del nivel 'n'
    for n in range(10):
        exprs_b2.append(
            (pl.col("bids[0].price") - pl.col(f"bids[{n}].price")).alias(
                f"v{61 + n}"
            )
        )

    # 4. Diferencias de volumen respecto a la punta - ASK (Variables 71 a 80)
    for n in range(10):
        exprs_b2.append(
            (pl.col(f"asks[{n}].amount") - pl.col("asks[0].amount")).alias(
                f"v{71 + n}"
            )
        )

    # 5. Diferencias de volumen respecto a la punta - BID (Variables 81 a 90)
    for n in range(10):
        exprs_b2.append(
            (pl.col(f"bids[{n}].amount") - pl.col("bids[0].amount")).alias(
                f"v{81 + n}"
            )
        )

    # -------------------------------------------------------------------------
    # BLOQUE 3: Características Temporales / Dinámicas (Variables 91 a 144)
    # -------------------------------------------------------------------------
    lookback = 5

    # 1. Diferencias de precio temporales (Variables 91 a 110) -> t menos (t-5)
    for n in range(10):
        exprs_b3.append(
            pl.col(f"asks[{n}].price").diff(lookback).alias(f"v{91 + n*2}")
        )
        exprs_b3.append(
            pl.col(f"bids[{n}].price").diff(lookback).alias(f"v{92 + n*2}")
        )

    # 2. Diferencias de volumen temporales (Variables 111 a 130) -> t menos (t-5)
    for n in range(10):
        exprs_b3.append(
            pl.col(f"asks[{n}].amount").diff(lookback).alias(f"v{111 + n*2}")
        )
        exprs_b3.append(
            pl.col(f"bids[{n}].amount").diff(lookback).alias(f"v{112 + n*2}")
        )

    # 3. Derivadas inmediatas (Variables 131 a 134) -> t menos (t-1)
    exprs_b3.append(pl.col("asks[0].price").diff(1).alias("v131"))
    exprs_b3.append(pl.col("bids[0].price").diff(1).alias("v132"))
    exprs_b3.append(pl.col("asks[0].amount").diff(1).alias("v133"))
    exprs_b3.append(pl.col("bids[0].amount").diff(1).alias("v134"))

    # 4. Ventanas móviles de intensidad del mercado (Variables 135 a 144)
    for i, window in enumerate([10, 20, 50]):
        base_idx = 135 + i * 3

        # Media móvil del spread de la punta
        exprs_b3.append(
            (pl.col("asks[0].price") - pl.col("bids[0].price"))
            .rolling_mean(window_size=window)
            .alias(f"v{base_idx}")
        )
        # Suma móvil del volumen total en la punta (Ask + Bid)
        exprs_b3.append(
            (pl.col("asks[0].amount") + pl.col("bids[0].amount"))
            .rolling_sum(window_size=window)
            .alias(f"v{base_idx+1}")
        )
        # Desviación estándar móvil del spread de la punta
        exprs_b3.append(
            (pl.col("asks[0].price") - pl.col("bids[0].price"))
            .rolling_std(window_size=window)
            .alias(f"v{base_idx+2}")
        )

    # -------------------------------------------------------------------------
    # BONUS EXTRAS (Guardamos el precio original y tu OBI)
    # -------------------------------------------------------------------------
    # Extraemos el mid-price real de la punta pero CON UN NOMBRE DE CONTROL
    # para que NO entre en la lista automática de features del XGBoost
    mid_price_expr = (
        (pl.col("asks[0].price") + pl.col("bids[0].price")) / 2.0
    ).alias("mid_price_coordenada")

    obi_expr = (
        (pl.col("bids[0].amount") - pl.col("asks[0].amount"))
        / (pl.col("bids[0].amount") + pl.col("asks[0].amount"))
    ).alias("OBI_Nivel_1")

    # Unimos todo el arsenal de Polars
    all_expressions = (
        exprs_b1 + exprs_b2 + exprs_b3 + [mid_price_expr, obi_expr]
    )

    # Procesamos todo en paralelo
    fi2010_df = df.select(all_expressions)

    return fi2010_df

In [9]:
X_features = extract_fi2010_features_strict_polars(df_book_10)

In [10]:



def compute_fi2010_labels_polars(
    df_input: pl.DataFrame, alpha: float = 0.002
) -> pl.DataFrame:
    """Calcula las etiquetas de predicción futuras (labels) según el método

    del paper FI-2010 solucionando el orden de evaluación paralela en Polars.
    """

    # 1. Calcular el Mid-Price actual (Punta del libro)
    df = df_input.with_columns(
        ((pl.col("asks[0].price") + pl.col("bids[0].price")) / 2.0).alias(
            "mid_price"
        )
    )

    horizons = [10, 20, 30, 50, 100]

    # PASO A: Crear primero las columnas de medias móviles futuras intermedias
    future_avg_exprs = [
        pl.col("mid_price")
        .shift(-k)
        .rolling_mean(window_size=k, min_samples=k)
        .alias(f"future_mid_avg_{k}")
        for k in horizons
    ]

    df = df.with_columns(future_avg_exprs)

    # PASO B: Ahora que las columnas existen seguro, calculamos los porcentajes y las etiquetas
    final_label_exprs = []
    for k in horizons:
        # Calcular cambio porcentual
        pct_change = (
            pl.col(f"future_mid_avg_{k}") - pl.col("mid_price")
        ) / pl.col("mid_price")

        # Clasificación en clases fijas para XGBoost (0, 1, 2)
        label_expr = (
            pl.when(pct_change >= alpha)
            .then(2)
            .when(pct_change <= -alpha)
            .then(0)
            .otherwise(1)
            .alias(f"label_k_{k}")
        )
        final_label_exprs.append(label_expr)

    # 2. Retornamos únicamente las columnas de etiquetas limpias
    return df.select(final_label_exprs)




In [11]:
y_labels = compute_fi2010_labels_polars(df_book_10, alpha=0.00002)

In [12]:
dataset_completo = pl.concat([X_features, y_labels], how="horizontal").drop_nulls()

In [13]:
dataset_completo.sink_parquet('dataset_completo.parquet')

In [14]:
num_filas = dataset_completo.select(pl.len()).collect().item()

print(f"El archivo tiene {num_filas:,} registros (filas)")

El archivo tiene 1,393,775 registros (filas)


In [15]:
print(y_labels.collect()["label_k_100"].value_counts())

shape: (3, 2)
┌─────────────┬────────┐
│ label_k_100 ┆ count  │
│ ---         ┆ ---    │
│ i32         ┆ u32    │
╞═════════════╪════════╡
│ 1           ┆ 440732 │
│ 0           ┆ 471554 │
│ 2           ┆ 481538 │
└─────────────┴────────┘
